In [0]:
import os
import math
import time
import requests
from datetime import datetime

from pyspark.sql.functions import col,count

In [0]:
file_id = "1AGXVlDhbMbhoGXDJG0IThnqz86Qy3hqb"
download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
local_path = "/dbfs/temp/transactions.csv"
s3_output_path = "dbfs:/mnt/s3mount/chunk/"
chunk_size = 10000

In [0]:
os.makedirs("local_path", exist_ok=True)

if not os.path.exists(local_path):
    r = requests.get(download_url)
    with open(local_path, "wb") as f:
        f.write(r.content)
        
schema= """step INT,
    customer STRING,
    age INT,           
    gender STRING,
    zipcodeOri STRING,
    merchant STRING,
    zipMerchant STRING,
    category STRING,
    amount DOUBLE,
    fraud INT"""

transactions_df = spark.read.option("header", "true").option("quote", "'").schema(schema).csv(f"file:{local_path}")

In [0]:
total_rows = transactions_df.count()
total_chunks = math.ceil(total_rows / chunk_size)
print(f"Total rows: {total_rows}, total chunks: {total_chunks}")

In [0]:
indexed_df = transactions_df.rdd.zipWithIndex().toDF()
indexed_df = indexed_df.selectExpr("_1.*", "_2 as row_index")
display(indexed_df)

In [0]:
for chunk_id in range(total_chunks):
    start = chunk_id * chunk_size
    end = start + chunk_size
    chunk_df = indexed_df.filter((col("row_index") >= start) & (col("row_index") < end)).drop("row_index")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    chunk_df.coalesce(1).write.mode("append").option("header", "true").csv(s3_output_path)
    print(f"Written chunk {chunk_id+1}/{total_chunks} to {s3_output_path}")
    time.sleep(1)

In [0]:
print('hi')